# Qwen3-32B Inference Benchmark on Amazon SageMaker AI

_Legal Disclaimer: This notebook is provided for demonstration and benchmarking purposes only. It is not intended for production use without additional security, compliance, and operational review. AWS customers are responsible for making their own assessment of the information in this document and any use of AWS services, each of which is provided "as is" without warranty of any kind, whether express or implied._

This notebook benchmarks **Qwen3-32B** inference on Amazon SageMaker AI using the native vLLM deep learning container across two GPU instance types: **ml.g6e.12xlarge** and **ml.g7e.2xlarge**.

**Dataset:** There is no dataset used for this benchmarking activity; just a workload consisting of a series of LLM prompts as defined in the cells below.

**Workload:** Technical documentation rescoring; token profile per inference call: ~1,000 input tokens (system prompt + dictation) → ~600 output tokens (corrected document).

We deploy using the [AWS vLLM DLC](https://github.com/aws/deep-learning-containers) (`vllm:0.15.1-gpu-py312-cu129`) with prefix caching and chunked prefill enabled.

## 1. Setup

Fetch the execution role and region from the SDK so we don't hardcode credentials.

In [ ]:
import boto3
import json
import sagemaker
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from datetime import datetime

sess = sagemaker.Session()
ROLE = sagemaker.get_execution_role()
REGION = sess.boto_region_name
sm = boto3.client("sagemaker", region_name=REGION)

PREFIX = "qwen3-qwen32b-v2"
VLLM_IMAGE = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/vllm:0.15.1-gpu-py312-cu129-ubuntu22.04-sagemaker" # AWS provided ECR repos

print(f"Region: {REGION}")
print(f"Role:   {ROLE}")
print(f"Image:  {VLLM_IMAGE}")

## 2. Endpoint Configurations

Each config maps to a specific instance type and vLLM tuning parameters. Key differences:

- **g6e** — 4× L40S with TP=4
- **g7e** — Single Blackwell GPU with TP=1

Both use BF16, prefix caching, and chunked prefill.

In [ ]:
CONFIGS = {
    "g6e": {
        "instance_type": "ml.g6e.12xlarge",
        "name_suffix": "g6e-vllm",
        "env": {
            "SM_VLLM_MODEL": "Qwen/Qwen3-32B",
            "SM_VLLM_TENSOR_PARALLEL_SIZE": "4",
            "SM_VLLM_DTYPE": "bfloat16",
            "SM_VLLM_MAX_MODEL_LEN": "4096",
            "SM_VLLM_ENABLE_PREFIX_CACHING": "",
            "SM_VLLM_ENABLE_CHUNKED_PREFILL": "",
            "SM_VLLM_MAX_NUM_SEQS": "128",
            "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9",
            "SM_VLLM_MAX_NUM_BATCHED_TOKENS": "16384",
        },
    },
    "g7e": {
        "instance_type": "ml.g7e.2xlarge",
        "name_suffix": "g7e-vllm",
        "env": {
            "SM_VLLM_MODEL": "Qwen/Qwen3-32B",
            "SM_VLLM_TENSOR_PARALLEL_SIZE": "1",
            "SM_VLLM_DTYPE": "bfloat16",
            "SM_VLLM_MAX_MODEL_LEN": "4096",
            "SM_VLLM_ENABLE_PREFIX_CACHING": "",
            "SM_VLLM_ENABLE_CHUNKED_PREFILL": "",
            "SM_VLLM_MAX_NUM_SEQS": "64",
            "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.9",
            "SM_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
        },
    },
}

## 3. Deploy Endpoints

The cell below checks for existing `InService` endpoints matching our naming prefix before deploying new ones. If endpoints are already running, it reuses them — avoiding duplicate deployments.

New deployments take ~10-15 minutes per endpoint for model download and loading.

In [ ]:
def find_existing_endpoint(suffix):
    """Find an InService endpoint matching our prefix and suffix."""
    paginator = sm.get_paginator("list_endpoints")
    for page in paginator.paginate(NameContains=PREFIX, StatusEquals="InService"):
        for ep in page["Endpoints"]:
            if suffix in ep["EndpointName"]:
                return ep["EndpointName"]
    return None


def deploy_endpoint(config_name):
    """Deploy a vLLM endpoint, or return existing one if already running."""
    cfg = CONFIGS[config_name]
    existing = find_existing_endpoint(cfg["name_suffix"])
    if existing:
        print(f"✅ Found existing endpoint: {existing}")
        return existing

    endpoint_name = f"{PREFIX}-{cfg['name_suffix']}-{datetime.now().strftime('%m%d-%H%M')}"
    print(f"🚀 Deploying {endpoint_name} on {cfg['instance_type']}...")

    model = Model(image_uri=VLLM_IMAGE, role=ROLE, predictor_cls=Predictor, env=cfg["env"])
    model.deploy(
        instance_type=cfg["instance_type"],
        initial_instance_count=1,
        endpoint_name=endpoint_name,
        container_startup_health_check_timeout=900,
        serializer=JSONSerializer(),
    )
    print(f"✅ Deployed: {endpoint_name}")
    return endpoint_name

In [ ]:
g6e_endpoint = deploy_endpoint("g6e")
g7e_endpoint = deploy_endpoint("g7e")

print(f"\ng6e: {g6e_endpoint}")
print(f"g7e: {g7e_endpoint}")

## 4. Benchmark Setup

We define our prompt with sample inputs to be used for benchmarking. The benchmark sweeps concurrency levels `[1, 2, 4, 8, 16, 32]` with 50 requests per level, measuring:
- **Latency** (p50, p90, p99)
- **Per-request throughput** (output tok/s)
- **Aggregate throughput** (total tok/s across concurrent requests)
- **RPS** (requests per second)

Data classification: the prompt contains sample mock data purely for the purposes of reaching the token profile limits.

In [ ]:
import time
import statistics
import concurrent.futures
from botocore.config import Config

SYSTEM_PROMPT = """You are a technical documentation rescoring and correction assistant. Your task is to take raw speech-to-text transcriptions from software engineers and produce a corrected, properly formatted technical document. You must:

1. Fix all technical terminology errors including API names, framework references, and architecture patterns
2. Correct service names, version numbers, configuration parameters, and CLI commands to standard notation
3. Maintain proper technical document formatting with appropriate sections and headers
4. Preserve the original technical meaning and intent of the dictating engineer
5. Follow standard technical writing guidelines including clear and concise language
6. Apply appropriate capitalization, punctuation, and paragraph structure
7. Expand common technical abbreviations where appropriate for clarity
8. Ensure system names, endpoints, configurations, and dependency lists are accurately transcribed
9. Flag any potentially incorrect configuration values or deprecated API references with [VERIFY] tags
10. Maintain consistency in naming conventions and terminology throughout the document

Output only the corrected technical document. Do not add commentary or explanations.""" + " ".join(["Technical documentation quality context and standardization guidelines."] * 30)

SAMPLE_INPUTS = [
    # Architecture design review
    "system design overview this is the architecture review for the order processing microservice we are migrating from the monolithic application to a distributed microservices architecture the current system handles approximately 50000 orders per day with peak loads reaching 200000 during promotional events the new architecture uses Amazon Elastic Container Service (Amazon ECS) with fargate launch type for compute we have three primary services the order intake service which receives requests via Amazon API Gateway and validates input schema against our openapi 3.0 specification the processing engine service which handles business logic including inventory checks pricing calculations and fraud detection using a rules engine built on apache flink and the fulfillment service which integrates with our warehouse management system via event bridge the services communicate asynchronously through Amazon Simple Queue Service (Amazon SQS) with dead letter queues configured for retry logic we use amazon dynamodb as the primary data store with single table design pattern the partition key is order id and we have a global secondary index on customer id for lookup queries read capacity is set to auto scaling with minimum 100 and maximum 5000 read capacity units write capacity follows similar auto scaling configuration for caching we implemented amazon elasticache redis cluster mode enabled with 3 shards and 2 replicas per shard the cache invalidation strategy uses write through for order status updates and time to live of 300 seconds for product catalog data we identified several technical risks first the dynamodb hot partition issue when a single customer places many orders during flash sales we plan to implement write sharding with a random suffix appended to the partition key second the sqs message ordering is not guaranteed so we added sequence numbers and implemented idempotency using conditional writes to dynamodb third cross region disaster recovery requires multi region dynamodb global tables with eventual consistency which means we need to handle conflict resolution at the application layer monitoring uses amazon cloudwatch with custom metrics for order processing latency error rates and queue depths we set alarms at p99 latency exceeding 500 milliseconds and error rate exceeding 1 percent the deployment pipeline uses aws codepipeline with blue green deployment strategy on ecs rolling updates take approximately 15 minutes with automated rollback triggered by cloudwatch alarms",
    # Sprint retrospective and planning
    "sprint 24 retrospective and sprint 25 planning meeting notes date march 15 2026 team alpha backend services attendance 8 of 9 team members present sprint 24 velocity was 42 story points against a planned 45 we completed 7 of 9 user stories with 2 stories carrying over to sprint 25 the authentication service migration to Amazon Cognito user pools was completed ahead of schedule the team delivered the oauth 2.0 authorization code flow with pkce support for mobile clients and implemented token refresh logic with sliding window expiration set to 30 minutes the api rate limiting feature using amazon Amazon API Gateway usage plans was also completed we configured three tiers free tier at 100 requests per day standard at 10000 requests per day and enterprise at unlimited with throttling at 500 requests per second the two incomplete stories were the Amazon OpenSearch Service to opensearch migration which is blocked by a version compatibility issue between opensearch 2.11 and our custom analyzer plugins and the graphql subscription implementation for real time order tracking which needs additional load testing what went well the pair programming sessions between junior and senior developers significantly improved code review turnaround time from an average of 2 days down to 4 hours the new automated integration test suite caught 3 critical bugs before they reached staging environment the team successfully reduced p99 api latency from 800 milliseconds to 320 milliseconds by implementing connection pooling and query optimization in the postgresql database what needs improvement deployment documentation is still scattered across multiple confluence pages and slack threads we need a single source of truth runbook inter team dependencies with the frontend team caused delays because api contract changes were not communicated early enough we should implement consumer driven contract testing using pact framework the on call rotation had 2 incidents this sprint both related to memory leaks in the node js services running on ecs we need to implement proper heap dump analysis and set memory limits in task definitions action items for sprint 25 create centralized runbook in github wiki deadline march 22 set up pact broker for contract testing deadline march 29 implement memory profiling with clinic js and set ecs task memory limits to 2048 megabytes with out of memory kill alerts sprint 25 planned stories total 44 points including the 2 carryovers estimated at 8 points each",
    # Infrastructure incident postmortem
    "post incident review incident number 2026 dash 0847 severity 2 duration 2 hours 37 minutes service affected customer facing Amazon API Gateway and downstream order processing pipeline date march 10 2026 14 colon 23 utc to 17 colon 00 utc impact approximately 12000 api requests failed with 503 service unavailable errors representing 8.3 percent of total traffic during the incident window estimated revenue impact 45000 dollars based on average order value and conversion rate timeline at 14 23 utc automated monitoring detected elevated 5xx error rates on the primary Amazon API Gateway endpoint cloudwatch alarm triggered when error rate exceeded 5 percent threshold the on call engineer was paged at 14 25 utc initial investigation showed that 3 of 6 ecs tasks in the order service cluster were in an unhealthy state the application load balancer health checks were failing with timeout errors at 14 42 utc the team identified that a configuration change deployed at 13 45 utc had modified the database connection pool settings the maximum connections parameter was changed from 20 to 200 per task which caused the aurora postgresql cluster to exceed its max connections limit of 500 this triggered connection queuing and eventual timeout cascading failures at 15 15 utc the team attempted to roll back the configuration change using the ecs service update command however the rollback was delayed because the previous task definition revision had been deregistered as part of a cleanup job that ran earlier that morning the team had to manually recreate the task definition with the correct parameters at 15 48 utc the corrected task definition was deployed and new tasks began starting at 16 30 utc all 6 tasks were healthy and error rates returned to baseline the team continued monitoring for 30 minutes before declaring the incident resolved at 17 00 utc root cause the database connection pool increase was intended to support higher concurrency but was not coordinated with the database team the Amazon Aurora instance class db r6g 2xlarge has a default max connections of 500 and with 6 tasks each opening 200 connections the total demand was 1200 connections exceeding capacity by 140 percent action items first implement connection pool validation in the cicd pipeline to check against known database limits deadline march 17 second add aurora connection count metric to the deployment canary analysis deadline march 20 third implement ecs task definition versioning policy that retains the last 5 revisions deadline march 24 fourth create a shared configuration management database for cross team infrastructure parameters deadline april 7 lessons learned configuration changes that affect shared infrastructure resources must go through a cross team review process the cleanup automation should never remove the immediately previous task definition revision and the runbook for database related incidents should include steps to check connection pool settings",
]

## 5. Benchmark Functions

Use the `ThreadPoolExecutor` library to send concurrent requests and compute latency/throughput benchmarks.

In [ ]:
smr = boto3.client("sagemaker-runtime", region_name=REGION, config=Config(read_timeout=300, retries={"max_attempts": 0}))

MAX_TOKENS = 600
CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 32]
REQUESTS_PER_LEVEL = 50


def invoke_endpoint(endpoint_name, prompt):
    """Single endpoint invocation with latency measurement."""
    payload = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        "max_tokens": MAX_TOKENS,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    start = time.perf_counter()
    try:
        resp = smr.invoke_endpoint(EndpointName=endpoint_name, ContentType="application/json", Body=json.dumps(payload))
        elapsed_ms = (time.perf_counter() - start) * 1000
        body = json.loads(resp["Body"].read())
        usage = body.get("usage", {})
        return {
            "success": True, "latency_ms": elapsed_ms,
            "input_tokens": usage.get("prompt_tokens", 0),
            "output_tokens": usage.get("completion_tokens", 0),
            "tok_per_sec": usage.get("completion_tokens", 0) / (elapsed_ms / 1000) if elapsed_ms > 0 else 0,
        }
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "error": str(e),
                "input_tokens": 0, "output_tokens": 0, "tok_per_sec": 0}


def run_benchmark(endpoint_name, concurrency, num_requests):
    """Run benchmark at a single concurrency level."""
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(invoke_endpoint, endpoint_name, SAMPLE_INPUTS[i % len(SAMPLE_INPUTS)]) for i in range(num_requests)]
        results = [f.result() for f in concurrent.futures.as_completed(futures)]

    ok = [r for r in results if r["success"]]
    if not ok:
        return {"concurrency": concurrency, "error": "All requests failed"}

    lats = sorted([r["latency_ms"] for r in ok])
    toks = [r["tok_per_sec"] for r in ok if r["tok_per_sec"] > 0]
    avg_lat_s = statistics.mean(lats) / 1000
    avg_out = statistics.mean([r["output_tokens"] for r in ok])
    avg_in = statistics.mean([r["input_tokens"] for r in ok])
    rps = concurrency / avg_lat_s if avg_lat_s > 0 else 0

    stats = {
        "concurrency": concurrency, "successful": len(ok), "failed": len(results) - len(ok),
        "latency_p50": lats[len(lats) // 2], "latency_p90": lats[int(len(lats) * 0.9)],
        "latency_p99": lats[int(len(lats) * 0.99)] if len(lats) >= 100 else lats[-1],
        "latency_avg": statistics.mean(lats),
        "tok_per_sec_avg": statistics.mean(toks) if toks else 0,
        "avg_input_tokens": avg_in, "avg_output_tokens": avg_out,
        "rps": rps, "aggregate_output_tok_sec": rps * avg_out,
    }
    print(f"  C={concurrency:>2d} | p50={stats['latency_p50']/1000:.1f}s | p99={stats['latency_p99']/1000:.1f}s | "
          f"tok/s={stats['tok_per_sec_avg']:.1f} | rps={rps:.2f}")
    return stats


def run_full_benchmark(endpoint_name):
    """Run benchmark across all concurrency levels with warmup."""
    print(f"Warming up {endpoint_name}...")
    for i in range(5):
        invoke_endpoint(endpoint_name, SAMPLE_INPUTS[0])
    print(f"Running benchmark: {endpoint_name}")
    all_stats = []
    for c in CONCURRENCY_LEVELS:
        all_stats.append(run_benchmark(endpoint_name, c, REQUESTS_PER_LEVEL))
        time.sleep(5)
    return all_stats

## 6. Run Benchmarks

Execute the full sweep for both endpoints. This will take ~20-30 minutes total (50 requests × 6 concurrency levels × 2 endpoints).

In [ ]:
g6e_results = run_full_benchmark(g6e_endpoint)
g7e_results = run_full_benchmark(g7e_endpoint)

## 7. Results Charts

Four publication-style charts comparing g6e vs g7e:
1. **p50 Latency** vs concurrency
2. **Per-request throughput** (output tok/s) vs concurrency
3. **Cost per million tokens** vs concurrency
4. **Aggregate output throughput** vs concurrency

In [ ]:
import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
import numpy as np

COST_PER_HR = {"g6e": 13.12, "g7e": 4.2039}

configs = [
    ("g6e vLLM (4×L40S)",  g6e_results, COST_PER_HR["g6e"], "tab:cyan"),
    ("g7e vLLM (Blackwell)", g7e_results, COST_PER_HR["g7e"], "tab:green"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for name, data, cost_hr, color in configs:
    conc = [r["concurrency"] for r in data if "error" not in r]
    valid = [r for r in data if "error" not in r]

    # p50 latency
    axes[0, 0].plot(conc, [r["latency_p50"] / 1000 for r in valid], "o--", label=name, color=color, markersize=5)
    # tok/s per request
    axes[0, 1].plot(conc, [r["tok_per_sec_avg"] for r in valid], "o--", label=name, color=color, markersize=5)
    # cost per M tokens
    costs = [(cost_hr / (r["rps"] * (r["avg_input_tokens"] + r["avg_output_tokens"]) * 3600)) * 1e6
             for r in valid if r["rps"] > 0]
    axes[1, 0].plot(conc[:len(costs)], costs, "o--", label=name, color=color, markersize=5)
    # aggregate throughput
    axes[1, 1].plot(conc, [r["aggregate_output_tok_sec"] for r in valid], "o--", label=name, color=color, markersize=5)

titles = ["p50 Latency", "Per-Request Throughput", "Cost per Million Tokens", "Aggregate Output Throughput"]
ylabels = ["Latency (seconds)", "Tokens/sec", "$/M tokens", "Total tokens/sec"]
for ax, title, ylabel in zip(axes.flat, titles, ylabels):
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Concurrent Clients")
    ax.set_ylabel(ylabel)
    ax.set_xticks([1, 2, 4, 8, 16, 32])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

fig.suptitle("Qwen3-32B on Amazon SageMaker AI — g6e vs g7e (vLLM, no EAGLE3)", fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

## 8. Summary Table

Best results at maximum concurrency (C=32).

In [ ]:
print(f"{'Config':<28} {'tok/s':>6} {'RPS':>6} {'p50 (s)':>8} {'$/M tok':>8}")
print("-" * 60)
for name, data, cost_hr, _ in configs:
    r = [d for d in data if d.get("concurrency") == 32]
    if not r:
        continue
    r = r[0]
    cost_m = (cost_hr / (r["rps"] * (r["avg_input_tokens"] + r["avg_output_tokens"]) * 3600)) * 1e6 if r["rps"] > 0 else 0
    print(f"{name:<28} {r['tok_per_sec_avg']:>6.1f} {r['rps']:>6.2f} {r['latency_p50']/1000:>8.1f} {cost_m:>7.2f}")

## License

This library is licensed under the MIT-0 License. See the LICENSE file.

## Security Responsibilities

AWS is responsible for securing the underlying Amazon SageMaker AI infrastructure, including the compute instances, hypervisor, and managed service components. Customers are responsible for: (1) configuring IAM roles and policies with least-privilege access, (2) enabling encryption for data at rest and in transit, (3) implementing network isolation through VPC configuration, (4) securing model artifacts and managing access controls, and (5) monitoring and logging endpoint activity. For details, see the AWS Shared Responsibility Model documentation.